# Heart Disease Risk Prediction: Exploratory Data Analysis & Audit

**Laboratory Project:** Artificial Intelligence and Expert Systems  
**Dataset:** Framingham Heart Study  
**Endpoint:** `TenYearCHD` (10-Year risk of developing coronary heart disease)  

---
### Academic Objectives:
1. Audit dataset completeness, data types, and missing value prevalence.
2. Screen for physiologically implausible values and clinical inconsistencies.
3. Inspect target class distribution and quantify class imbalance.
4. Evaluate feature correlations and distributions prior to feature selection.
5. Enforce data hygiene: **No data leakage, no premature oversampling, and no target contamination.**

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RAW_DATA_PATH, TARGET_COLUMN, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, PHYSIOLOGICAL_BOUNDS
from src.data_audit import DataAuditor, DataAuditException

print(f"Project root: {project_root}")
print(f"Target data path: {RAW_DATA_PATH}")

## 1. File Inspection & Safe Ingestion

Inspect the physical CSV file before attempting to load it to guard against empty placeholders or missing target columns.

In [ ]:
auditor = DataAuditor()
status = auditor.inspect_file_status()
print('File status:', status)

if status['is_empty']:
    print('[WARNING] File is 0 bytes. Authentic Framingham data must be populated.')
else:
    df = auditor.load_dataset()
    print(f'Loaded dataset with shape: {df.shape}')
    display(df.head())

## 2. Statistical and Clinical Audit

Execute the comprehensive audit examining missingness, duplicates, class distributions, and physiological boundaries.

In [ ]:
try:
    df = auditor.load_dataset()
    audit_results = auditor.run_full_audit(df)
    print(f"Total records: {audit_results['total_records']}")
    print(f"Complete cases: {audit_results['complete_cases']} ({audit_results['complete_case_pct']}%)")
    print(f"Class distribution: {audit_results['target_distribution']['counts']}")
    print(f"Imbalance ratio: {audit_results['target_distribution']['imbalance_ratio']}")
except DataAuditException as e:
    print(f"Audit halted safely: {e}")

## 3. Visualizations & Report Generation

Generate all figures to `reports/figures/` and compile `reports/eda_report.md`.

In [ ]:
try:
    df = auditor.load_dataset()
    saved_figs = auditor.generate_visualizations(df)
    for fig_path in saved_figs:
        print(f"Saved figure: {fig_path.name}")
    report_path = auditor.generate_markdown_report(audit_results)
    print(f"Generated report: {report_path}")
except DataAuditException as e:
    print(f"Visualizations cannot be generated on an empty dataset: {e}")